In [63]:
!uv sync
!source ../venv/bin/activate
!uv add pandas


Resolved 70 packages in 17ms
Audited 66 packages in 20ms
Resolved 70 packages in 4ms
Audited 65 packages in 1ms


In [1]:
import json
import glob
from pathlib import Path
import pandas as pd


In [6]:
def read_json(filename):
    with open(filename, newline='') as f:
        data = json.load(f)
    return data

results_path = Path("../results/")
timestamp_dirs = [d for d in results_path.iterdir() if d.is_dir()]

all_results = []

# Get the last 15 timestamp directories (or all if less than 15)
num_dirs_to_process = min(15, len(timestamp_dirs))
if timestamp_dirs:
    # Sort by directory name (timestamp) and get the last 15
    sorted_dirs = sorted(timestamp_dirs, key=lambda x: x.name)
    last_n_dirs = sorted_dirs[-num_dirs_to_process:]
    
    print(f"Processing {len(last_n_dirs)} timestamp directories:")
    for dir_path in last_n_dirs:
        print(f"  - {dir_path.name}")
    
    # Process each timestamp directory
    for timestamp_dir in last_n_dirs:
        print(f"\nProcessing: {timestamp_dir.name}")
        
        for item in timestamp_dir.iterdir():
            quantum_computer = item.name
            if item.is_dir():
                for optimising_algorithm in item.iterdir():
                    if optimising_algorithm.is_dir():
                        for mapping_algorithm in optimising_algorithm.iterdir():
                            if mapping_algorithm.is_file() and mapping_algorithm.suffix == ".json":
                                data = read_json(mapping_algorithm)
                                enriched_result = {
                                    'timestamp': timestamp_dir.name,
                                    'topology': quantum_computer,
                                    'algorithm': optimising_algorithm.name,
                                    'optimizer': mapping_algorithm.name,
                                    'swap_count': data.get('swap_count', None),
                                    'cx_count': data.get('cx_count', None),
                                    'depth': data.get('depth', None),
                                    'optimisation_time': data.get('optimisation_time', None),
                                    'file_path': str(mapping_algorithm)
                                }
                                all_results.append(enriched_result)
    
    print(f"\n✓ Collected {len(all_results)} total results from {len(last_n_dirs)} runs")
    
    # Create dataframe with all results
    df_all = pd.DataFrame(all_results)
    
    # Calculate median values for each topology/algorithm/optimizer combination
    print("\nCalculating median values for each configuration...")
    
    # Group by topology, algorithm, optimizer and calculate median
    df_median = df_all.groupby(['topology', 'algorithm', 'optimizer']).agg({
        'swap_count': 'median',
        'cx_count': 'median',
        'depth': 'median',
        'optimisation_time': 'median'
    }).reset_index()
    
    # Add a count column to see how many runs were averaged
    df_counts = df_all.groupby(['topology', 'algorithm', 'optimizer']).size().reset_index(name='run_count')
    df_median = df_median.merge(df_counts, on=['topology', 'algorithm', 'optimizer'])
    
    print(f"✓ Created median dataframe with {len(df_median)} unique configurations")
    print(f"\nSample of median results:")
    print(df_median.head(10))
    
else:
    print("No timestamp directories found")
    df_all = pd.DataFrame()
    df_median = pd.DataFrame()


Processing 15 timestamp directories:
  - 1768078368
  - 1768166247
  - 1768167997
  - 1768169078
  - 1768169793
  - 1768170479
  - 1768171235
  - 1768173313
  - 1768225827
  - 1768226661
  - 1768227674
  - 1768228986
  - 1768230465
  - 1768231411
  - 1768232816

Processing: 1768078368

Processing: 1768166247

Processing: 1768167997

Processing: 1768169078

Processing: 1768169793

Processing: 1768170479

Processing: 1768171235

Processing: 1768173313

Processing: 1768225827

Processing: 1768226661

Processing: 1768227674

Processing: 1768228986

Processing: 1768230465

Processing: 1768231411

Processing: 1768232816

✓ Collected 8640 total results from 15 runs

Calculating median values for each configuration...
✓ Created median dataframe with 576 unique configurations

Sample of median results:
             topology      algorithm                  optimizer  swap_count  \
0  google_willow_q105    demo_random               doustra.json       175.0   
1  google_willow_q105    demo_random 

In [8]:
# Use the median dataframe for analysis
df = df_median.copy()

print("Median Results DataFrame:")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nRun counts per configuration:")
print(df['run_count'].describe())


Median Results DataFrame:
Shape: (576, 8)

Columns: ['topology', 'algorithm', 'optimizer', 'swap_count', 'cx_count', 'depth', 'optimisation_time', 'run_count']

First few rows:
             topology    algorithm                  optimizer  swap_count  \
0  google_willow_q105  demo_random               doustra.json       175.0   
1  google_willow_q105  demo_random      lightSABRE_decay.json         6.0   
2  google_willow_q105  demo_random  lightSABRE_lookahead.json         6.0   
3  google_willow_q105  demo_random           pauliforest.json        11.0   
4  google_willow_q105  demo_random             qiskit_ai.json        40.0   

   cx_count  depth  optimisation_time  run_count  
0     590.0  190.0           0.702531         15  
1      74.0  127.0           0.034199         15  
2      74.0  128.0           0.221424         15  
3      78.0   46.0           0.468280         15  
4     180.0  316.0           0.141433         15  

Run counts per configuration:
count    576.0
mean    

In [4]:
# Get tables according to the algorithm
if len(df) > 0:
    for algorithm in df['algorithm'].unique():
        # Filter data for this algorithm
        algorithm_df = df[df['algorithm'] == algorithm]
        
        print(f"\n Algorithm: {algorithm.upper()}")
        print("-" * 60)
        
        display_columns = ['topology', 'optimizer', 'swap_count', 'depth', 'optimisation_time']
        algorithm_table = algorithm_df[display_columns].copy()
        
        # Format the optimization time to show in milliseconds
        algorithm_table['optimisation_time_ms'] = algorithm_table['optimisation_time'].apply(
            lambda x: f"{x*1000:.2f}" if x is not None else "N/A"
        )
        
        # Drop the original time column and rename
        algorithm_table = algorithm_table.drop('optimisation_time', axis=1)
        algorithm_table = algorithm_table.rename(columns={
            'topology': 'Topology',
            'optimizer': 'Optimizer',
            'swap_count': 'SWAP Count',
            'cx_count': 'CNOT Count',
            'depth': 'Circuit Depth',
            'optimisation_time_ms': 'Time (ms)'
        })
        
        # Display the table
        print(algorithm_table.to_string(index=False))
        
        # Add summary statistics
        numeric_cols = algorithm_df[['swap_count', 'depth', 'optimisation_time']].select_dtypes(include=['number'])
        if not numeric_cols.empty:
            print(f"\n Summary for {algorithm}:")
            print(f"   • Average SWAP count: {algorithm_df['swap_count'].mean():.2f}")
            print(f"   • Average circuit depth: {algorithm_df['depth'].mean():.2f}")
            print(f"   • Average optimization time: {algorithm_df['optimisation_time'].mean()*1000:.2f} ms")
            
            # Find best optimizer for each metric
            if len(algorithm_df) > 1:
                best_swap = algorithm_df.loc[algorithm_df['swap_count'].idxmin()]
                best_depth = algorithm_df.loc[algorithm_df['depth'].idxmin()]
                best_time = algorithm_df.loc[algorithm_df['optimisation_time'].idxmin()]
                
                print(f"   • Best SWAP count: {best_swap['optimizer']} ({best_swap['swap_count']} swaps)")
                print(f"   • Best circuit depth: {best_depth['optimizer']} ({best_depth['depth']} gates)")
                print(f"   • Fastest optimization: {best_time['optimizer']} ({best_time['optimisation_time']*1000:.2f} ms)")
        
        print("\n" + "=" * 80)



 Algorithm: EFFICIENT_SU2
------------------------------------------------------------
             Topology                 Optimizer  SWAP Count  Circuit Depth Time (ms)
       ibm_falcon_q27     lightSABRE_decay.json           0             12      8.58
       ibm_falcon_q27 lightSABRE_lookahead.json           0             19     24.64
       ibm_falcon_q27               rustiq.json           0             12      6.15
       ibm_falcon_q27              doustra.json          17             56    208.66
       ibm_falcon_q27          pauliforest.json          -1             -1      0.00
       ibm_falcon_q27            qiskit_ai.json           2             29     78.12
       ibm_eagle_q127     lightSABRE_decay.json           0             16     13.07
       ibm_eagle_q127 lightSABRE_lookahead.json           0             12     31.35
       ibm_eagle_q127               rustiq.json           0             13     11.80
       ibm_eagle_q127              doustra.json          16   

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

def escape_latex(text):
    """Escape special LaTeX characters, particularly underscores."""
    return str(text).replace('_', r'\_')

def format_time_comma(time_ms):
    """Format time with comma as decimal separator."""
    return f"{time_ms:.2f}".replace('.', ',')

def generate_latex_tables(df, use_longtable=False, topologies=None):
    """
    Generate LaTeX tables for each algorithm, optimized for A4 page.
    Includes CNOT count (cx_count).
    Sorted by Topology, then Alphabetically by Optimizer.
    """
    
    latex_output = []
    
    # Filter topologies if specified
    if topologies is not None:
        df = df[df['topology'].isin(topologies)].copy()
    
    for algorithm in sorted(df['algorithm'].unique()):
        algorithm_df = df[df['algorithm'] == algorithm].copy()
        
        algorithm_df = algorithm_df.sort_values(['topology', 'optimizer'])
        
        # Format algorithm name for display
        algo_display = escape_latex(algorithm.replace('_', ' ').title())
        
        # Create section
        latex_output.append(f"\n\\section*{{{algo_display}}}\n")
        
        # Prepare data
        table_data = algorithm_df[['topology', 'optimizer', 'swap_count', 'cx_count', 'depth', 'optimisation_time']].copy()
        
        # Format optimizer names
        table_data['optimizer'] = table_data['optimizer'].str.replace('.json', '', regex=False)
        
        # Convert optimization time to milliseconds and round
        table_data['optimisation_time'] = (table_data['optimisation_time'] * 1000).round(2)
        
        # Group by topology (maintains sort order from algorithm_df)
        topo_list = table_data['topology'].unique()
        
        # Start table
        if use_longtable:
            latex_output.append(r"\begin{longtable}{|l|l|r|r|r|r|}")
            latex_output.append(r"\caption{" + f"Results for {algo_display} algorithm" + r"} \\")
            latex_output.append(f"\\label{{tab:{algorithm}_full}} \\\\")
            latex_output.append(r"\toprule")
            latex_output.append(r"Topology & Optimizer & SWAP count & CNOT count & Depth & Time (ms) \\")
            latex_output.append(r"\midrule")
            latex_output.append(r"\endfirsthead")
            latex_output.append(r"\multicolumn{6}{c}{{\tablename\ \thetable{} -- continued from previous page}} \\")
            latex_output.append(r"\toprule")
            latex_output.append(r"Topology & Optimizer & SWAP count & CNOT count & Depth & Time (ms) \\")
            latex_output.append(r"\midrule")
            latex_output.append(r"\endhead")
            latex_output.append(r"\midrule")
            latex_output.append(r"\multicolumn{6}{r}{{Continued on next page}} \\")
            latex_output.append(r"\endfoot")
            latex_output.append(r"\bottomrule")
            latex_output.append(r"\endlastfoot")
        else:
            latex_output.append(r"\begin{table}[htbp]")
            latex_output.append(r"\centering")
            latex_output.append(r"\adjustbox{max width=\textwidth}{")
            latex_output.append(r"\begin{tabular}{l|l|r|r|r|r}")
            latex_output.append(r"\toprule")
            latex_output.append(r"Topology & Optimizer & SWAP count & CNOT count & Depth & Time (ms) \\")
            latex_output.append(r"\midrule")
        
        for i, topology in enumerate(topo_list):
            # topo_data will be sorted by optimizer because table_data was sorted
            topo_data = table_data[table_data['topology'] == topology]
            
            # Format topology name
            topo_display = escape_latex(topology.replace('_', ' ').title())
            
            for j, (_, row) in enumerate(topo_data.iterrows()):
                optimizer_escaped = escape_latex(row['optimizer'])
                
                if j == 0:
                    latex_output.append(
                        f"\\multirow{{{len(topo_data)}}}{{*}}{{{topo_display}}} & "
                        f"{optimizer_escaped} & "
                        f"{int(row['swap_count'])} & "
                        f"{int(row['cx_count'])} & "
                        f"{int(row['depth'])} & "
                        f"{format_time_comma(row['optimisation_time'])} \\\\"
                    )
                else:
                    latex_output.append(
                        f"& {optimizer_escaped} & "
                        f"{int(row['swap_count'])} & "
                        f"{int(row['cx_count'])} & "
                        f"{int(row['depth'])} & "
                        f"{format_time_comma(row['optimisation_time'])} \\\\"
                    )
            
            if i < len(topo_list) - 1:
                latex_output.append(r"\hline")
        
        # End table
        if use_longtable:
            latex_output.append(r"\end{longtable}")
        else:
            latex_output.append(r"\bottomrule")
            latex_output.append(r"\end{tabular}")
            latex_output.append(r"}")
            latex_output.append(f"\\caption{{Results for {algo_display} algorithm}}")
            latex_output.append(f"\\label{{tab:{algorithm}}}")
            latex_output.append(r"\end{table}")
        
        # Add summary statistics table
        latex_output.append("\n\\vspace{1em}\n")
        latex_output.append(r"\begin{table}[htbp]")
        latex_output.append(r"\centering")
        latex_output.append(f"\\caption{{Summary statistics for {algo_display} algorithm}}")
        latex_output.append(f"\\label{{tab:{algorithm}_summary}}")
        latex_output.append(r"\begin{tabular}{lrrrrr}")
        latex_output.append(r"\toprule")
        latex_output.append(r"Optimizer & Avg SWAP & Avg CNOT & Avg depth & Avg time (ms) & Count \\")
        latex_output.append(r"\midrule")
        
        # Ensure summary is also sorted alphabetically
        for optimizer in sorted(algorithm_df['optimizer'].str.replace('.json', '', regex=False).unique()):
            opt_data = algorithm_df[algorithm_df['optimizer'].str.replace('.json', '', regex=False) == optimizer]
            
            avg_swap = opt_data['swap_count'].mean()
            avg_cx = opt_data['cx_count'].mean()
            avg_depth = opt_data['depth'].mean()
            avg_time = opt_data['optimisation_time'].mean() * 1000
            count = len(opt_data)
            
            latex_output.append(
                f"{escape_latex(optimizer)} & "
                f"{avg_swap:.1f}".replace('.', ',') + " & "
                f"{avg_cx:.1f}".replace('.', ',') + " & "
                f"{avg_depth:.1f}".replace('.', ',') + " & "
                f"{format_time_comma(avg_time)} & "
                f"{count} \\\\"
            )
        
        latex_output.append(r"\bottomrule")
        latex_output.append(r"\end{tabular}")
        latex_output.append(r"\end{table}")
        
        latex_output.append("\n\\clearpage\n")
    
    return '\n'.join(latex_output)


def generate_compact_latex_tables(df, use_longtable=False, topologies=None):
    """
    Generate more compact LaTeX tables with abbreviated topology names.
    Sorted by Device, then Alphabetically by Optimizer.
    """
    
    latex_output = []
    
    topo_abbrev = {
        'ibm_falcon_q27': 'Falcon-27',
        'ibm_eagle_q127': 'Eagle-127',
        'ionq_harmony_q9': 'Harmony-9',
        'ibm_manhattan_q65': 'Manhattan-65',
        'ibm_reuschlikon_q16': 'Rueschlikon-16',
        'rigetti_novera_q9': 'Novera-9',
        'hexagonal_lattice_q54': 'Hex-54',
        'google_willow_q105': 'Willow-105',
        'ibm_heron_q133': 'Heron-133',
        'riken_fujitsu_q256': 'Riken-256',
        'ibm_almaden_q20': 'Almaden-20',
        'ibm_montreal_q27': 'Montreal-27',
        'ibm_cambridge_q28': 'Cambridge-28',
        'ibm_tokyo_q20': 'Tokyo-20',
        'ibm_paughkeepsie_q20': 'Poughkeepsie-20',
        'ibm_rochester_q53': 'Rochester-53'
    }
    
    if topologies is not None:
        df = df[df['topology'].isin(topologies)].copy()
    
    for algorithm in sorted(df['algorithm'].unique()):
        algorithm_df = df[df['algorithm'] == algorithm].copy()
        
        # --- MODIFICATION: Sort by topology AND optimizer ---
        algorithm_df = algorithm_df.sort_values(['topology', 'optimizer'])
        
        algo_display = escape_latex(algorithm.replace('_', ' ').upper())
        
        latex_output.append(f"\n\\section*{{{algo_display}}}\n")
        
        if use_longtable:
            latex_output.append(r"\begin{longtable}{|l|l|r|r|r|r|}")
            latex_output.append(r"\caption{" + f"Results for {algo_display} algorithm" + r"} \\")
            latex_output.append(f"\\label{{tab:{algorithm}_full}} \\\\")
            latex_output.append(r"\toprule")
            latex_output.append(r"Device & Optimizer & SWAP & CNOT & Depth & Time (ms) \\")
            latex_output.append(r"\midrule")
            latex_output.append(r"\endfirsthead")
            latex_output.append(r"\multicolumn{6}{c}{{\tablename\ \thetable{} -- continued from previous page}} \\")
            latex_output.append(r"\toprule")
            latex_output.append(r"Device & Optimizer & SWAP & CNOT & Depth & Time (ms) \\")
            latex_output.append(r"\midrule")
            latex_output.append(r"\endhead")
            latex_output.append(r"\midrule")
            latex_output.append(r"\multicolumn{6}{r}{{Continued on next page}} \\")
            latex_output.append(r"\endfoot")
            latex_output.append(r"\bottomrule")
            latex_output.append(r"\endlastfoot")
        else:
            latex_output.append(r"\begin{table}[htbp]")
            latex_output.append(r"\centering")
            latex_output.append(r"\small")
            latex_output.append(r"\begin{tabular}{|l|l|r|r|r|r|}")
            latex_output.append(r"\toprule")
            latex_output.append(r"Device & Optimizer & SWAP & CNOT & Depth & Time (ms) \\")
            latex_output.append(r"\midrule")
        
        topo_list = algorithm_df['topology'].unique()
        
        for i, topology in enumerate(topo_list):
            topo_data = algorithm_df[algorithm_df['topology'] == topology]
            topo_abbr = topo_abbrev.get(topology, escape_latex(topology))
            
            for j, (_, row) in enumerate(topo_data.iterrows()):
                optimizer = row['optimizer'].replace('.json', '').replace('pauliforest', 'pforest')
                optimizer_escaped = escape_latex(optimizer)
                
                if j == 0:
                    latex_output.append(
                        f"\\multirow{{{len(topo_data)}}}{{*}}{{{topo_abbr}}} & "
                        f"{optimizer_escaped} & "
                        f"{int(row['swap_count'])} & "
                        f"{int(row['cx_count'])} & "
                        f"{int(row['depth'])} & "
                        f"{format_time_comma(row['optimisation_time']*1000)} \\\\"
                    )
                else:
                    latex_output.append(
                        f"& {optimizer_escaped} & "
                        f"{int(row['swap_count'])} & "
                        f"{int(row['cx_count'])} & "
                        f"{int(row['depth'])} & "
                        f"{format_time_comma(row['optimisation_time']*1000)} \\\\"
                    )
            
            if i < len(topo_list) - 1:
                latex_output.append(r"\hline")
        
        if use_longtable:
            latex_output.append(r"\end{longtable}")
        else:
            latex_output.append(r"\bottomrule")
            latex_output.append(r"\end{tabular}")
            latex_output.append(f"\\caption{{Results for {algo_display} algorithm}}")
            latex_output.append(f"\\label{{tab:{algorithm}}}")
            latex_output.append(r"\end{table}")
        
        latex_output.append("\n")
    
    return '\n'.join(latex_output)


def generate_single_algorithm_table(algo_df, algorithm, use_longtable=False):
    """
    Generate a table for a single algorithm.
    Sorted by Topology, then Alphabetically by Optimizer.
    """
    latex_output = []
    
    algorithm_df = algo_df.copy()
    
    algorithm_df = algorithm_df.sort_values(['topology', 'optimizer'])
    
    algo_display = escape_latex(algorithm.replace('_', ' ').title())
    
    table_data = algorithm_df[['topology', 'optimizer', 'swap_count', 'cx_count', 'depth', 'optimisation_time']].copy()
    table_data['optimizer'] = table_data['optimizer'].str.replace('.json', '', regex=False)
    table_data['optimisation_time'] = (table_data['optimisation_time'] * 1000).round(2)
    
    topo_list = table_data['topology'].unique()
    
    if use_longtable:
        latex_output.append(r"\begin{longtable}{|l|l|r|r|r|r|}")
        latex_output.append(r"\caption{" + f"Results for {algo_display} algorithm" + r"} \\")
        latex_output.append(f"\\label{{tab:{algorithm}_full}} \\\\")
        latex_output.append(r"\toprule")
        latex_output.append(r"Topology & Optimizer & SWAP & CNOT & Depth & Time (ms) \\")
        latex_output.append(r"\midrule")
        latex_output.append(r"\endfirsthead")
        latex_output.append(r"\multicolumn{6}{c}{{\tablename\ \thetable{} -- continued from previous page}} \\")
        latex_output.append(r"\toprule")
        latex_output.append(r"Topology & Optimizer & SWAP & CNOT & Depth & Time (ms) \\")
        latex_output.append(r"\midrule")
        latex_output.append(r"\endhead")
        latex_output.append(r"\midrule")
        latex_output.append(r"\multicolumn{6}{r}{{Continued on next page}} \\")
        latex_output.append(r"\endfoot")
        latex_output.append(r"\bottomrule")
        latex_output.append(r"\endlastfoot")
    else:
        latex_output.append(r"\begin{table}[htbp]")
        latex_output.append(r"\centering")
        latex_output.append(r"\small")
        latex_output.append(r"\begin{tabular}{|l|l|r|r|r|r|}")
        latex_output.append(r"\toprule")
        latex_output.append(r"Topology & Optimizer & SWAP & CNOT & Depth & Time (ms) \\")
        latex_output.append(r"\midrule")
    
    for i, topology in enumerate(topo_list):
        topo_data = table_data[table_data['topology'] == topology]
        topo_display = escape_latex(topology.replace('_', ' ').title())
        
        for j, (_, row) in enumerate(topo_data.iterrows()):
            optimizer_escaped = escape_latex(row['optimizer'])
            
            if j == 0:
                latex_output.append(
                    f"\\multirow{{{len(topo_data)}}}{{*}}{{{topo_display}}} & "
                    f"{optimizer_escaped} & "
                    f"{int(row['swap_count'])} & "
                    f"{int(row['cx_count'])} & "
                    f"{int(row['depth'])} & "
                    f"{format_time_comma(row['optimisation_time'])} \\\\"
                )
            else:
                latex_output.append(
                    f"& {optimizer_escaped} & "
                    f"{int(row['swap_count'])} & "
                    f"{int(row['cx_count'])} & "
                    f"{int(row['depth'])} & "
                    f"{format_time_comma(row['optimisation_time'])} \\\\"
                )
        
        if i < len(topo_list) - 1:
            latex_output.append(r"\hline")
    
    if use_longtable:
        latex_output.append(r"\end{longtable}")
    else:
        latex_output.append(r"\bottomrule")
        latex_output.append(r"\end{tabular}")
        latex_output.append(f"\\caption{{Results for {algo_display} algorithm}}")
        latex_output.append(f"\\label{{tab:{algorithm}}}")
        latex_output.append(r"\end{table}")
    
    return '\n'.join(latex_output)


def save_tables_individually(df, output_dir='latex_tables', use_longtable=False, topologies=None):
    """Save each algorithm's table to a separate file."""
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    if topologies is not None:
        df = df[df['topology'].isin(topologies)].copy()
        print(f"Filtering to {len(topologies)} topologies")
    
    print(f"Generating individual table files in '{output_dir}/'...")
    
    for algorithm in sorted(df['algorithm'].unique()):
        algo_df = df[df['algorithm'] == algorithm]
        table_content = generate_single_algorithm_table(algo_df, algorithm, use_longtable)
        
        filename = output_path / f"table_{algorithm}.tex"
        with open(filename, 'w') as f:
            f.write(table_content)
        
        print(f"✓ Saved: {filename}")
    
    # Create master include file
    master_content = ["% Include all algorithm tables", ""]
    for algorithm in sorted(df['algorithm'].unique()):
        master_content.append(f"\\input{{{output_dir}/table_{algorithm}.tex}}")
        master_content.append("")
    
    with open(output_path / "all_tables.tex", 'w') as f:
        f.write('\n'.join(master_content))
    
    print(f"✓ Saved master file: {output_path / 'all_tables.tex'}")



save_tables_individually(df, output_dir='latex_tables', use_longtable=False)

save_tables_individually(df, output_dir='latex_tables_long', use_longtable=True)


subset_topologies = ['rigetti_novera_q9', 'ibm_eagle_q127', 'google_willow_q105', 'riken_fujitsu_q256']
save_tables_individually(df, output_dir='latex_tables_subset', 
                        use_longtable=False, topologies=subset_topologies)



Generating individual table files in 'latex_tables/'...
✓ Saved: latex_tables/table_demo_random.tex
✓ Saved: latex_tables/table_efficient_su2.tex
✓ Saved: latex_tables/table_h2.tex
✓ Saved: latex_tables/table_liH.tex
✓ Saved: latex_tables/table_qaoa.tex
✓ Saved: latex_tables/table_qaoa_ansatz.tex
✓ Saved master file: latex_tables/all_tables.tex
Generating individual table files in 'latex_tables_long/'...
✓ Saved: latex_tables_long/table_demo_random.tex
✓ Saved: latex_tables_long/table_efficient_su2.tex
✓ Saved: latex_tables_long/table_h2.tex
✓ Saved: latex_tables_long/table_liH.tex
✓ Saved: latex_tables_long/table_qaoa.tex
✓ Saved: latex_tables_long/table_qaoa_ansatz.tex
✓ Saved master file: latex_tables_long/all_tables.tex
Filtering to 4 topologies
Generating individual table files in 'latex_tables_subset/'...
✓ Saved: latex_tables_subset/table_demo_random.tex
✓ Saved: latex_tables_subset/table_efficient_su2.tex
✓ Saved: latex_tables_subset/table_h2.tex
✓ Saved: latex_tables_subset/ta